In [356]:
import pandas as pd
import ast

PATH = "../data/raw/listings_raw.csv"
REPORTS = "../reports/"
OUTPUT = "../data/raw/listings.csv"


In [357]:
df = pd.read_csv(PATH)

In [358]:
df.isna().sum()

url               0
description       0
fetch_date        0
title             0
location          0
house_type       18
bathrooms         1
bedrooms          1
properties        0
amenities      2233
price             0
dtype: int64

In [359]:
def get_loc(loc: pd.DataFrame):
    parts = str(loc).split(",")
    if len(parts) >= 3:
        return parts[1].strip()
    return parts[0]


In [360]:
df["location"] = df["location"].apply(get_loc)

In [361]:
def fix_ht(df):
    mht = df["house_type"].isna().sum()
    print(f"Missing house_type data before update: {mht}")
    df.loc[df["house_type"].isna(), "house_type"] = "Bedsitter"
    mht = df["house_type"].isna().sum()
    print(f"Missing house_type data after update nans: {mht}")
    return df

In [362]:
df = fix_ht(df)

Missing house_type data before update: 18
Missing house_type data after update nans: 0


In [363]:
def fix_bednbath(df):
    print(f"Missing bathroom data before update: {df['bathrooms'].isna().sum()}")
    print(f"Missing bedroom data before update: {df['bedrooms'].isna().sum()}")

    df.dropna(subset=["bathrooms"], inplace=True)
    df.dropna(subset=["bedrooms"], inplace=True)

    df["bathrooms"] = df["bathrooms"].str.split(" ").str.get(0).str.strip()
    df["bedrooms"] = df["bedrooms"].str.split(" ").str.get(0).str.strip()

    print(f"Missing bathroom data after update: {df['bathrooms'].isna().sum()}")
    print(f"Missing bedroom data after update: {df['bedrooms'].isna().sum()}")

    return df

In [364]:
df = fix_bednbath(df)

Missing bathroom data before update: 1
Missing bedroom data before update: 1
Missing bathroom data after update: 0
Missing bedroom data after update: 0


In [365]:
def extract_all_attributes(df, drop=True):
    """Expand the property column into separate columns"""
    olen = len(df.columns)

    df["properties"] = df["properties"].apply(ast.literal_eval)
    properties = df["properties"].apply(pd.Series)
    df_prop = df.join(properties)
    if drop:
        df_prop = df_prop.drop("properties", axis=1)

    plen = len(df_prop.columns)

    print(f"Properties extracted; added {plen - olen} new attributes!")

    amenities = df_prop["amenities"].str.strip().str.get_dummies(sep=",")
    df_amen = df_prop.join(amenities)
    if drop:
        df_amen = df_amen.drop("amenities", axis=1)

    alen = len(df_amen.columns)
    print(f"Ameneties extracted; added {alen - plen} new attributes!")
    print(f"{alen} final attributes after extraction")

    facilities = df_amen["Facilities"].str.strip().str.get_dummies(sep=",")
    df_amen.update(facilities)  # Updates overlapping columns in-place
    df_fac = df_amen

    return df_fac

In [366]:
df = extract_all_attributes(df)

Properties extracted; added 23 new attributes!
Ameneties extracted; added 17 new attributes!
51 final attributes after extraction


In [367]:
df["price"] = df["price"].str.replace("GH₵ ", "").str.replace(",", "").astype(float)

In [ ]:
def clean_self_contained(df):
    """
    Clean and fix the 'Self Contained' column based on sequential rules.
    """
    df = df.copy()

    print(f"NAs in self contained feature: {len(df[df['Self Contained'].isna()])}")

    search_terms = [
        "Self Contained",
        "self-contained",
        "self-compound",
        "hall sc",
        "S/C",
        "self apartment",
        "acs",
        "aircondition",
        "self contain",
        "exacutive",
        "Self Compound",
        "Luxury",
        "Newly",
        "Executive",
        "exective",
        "ultra",
        "modern",
        "furnished",
        "new",
        "suite",
        "ensuit",
        "en suite",
        "en-suite",
        "USD",
        "dollar",
        r"\$",
        "excellent",
        "Air condition",
        "Air-condition",
        "conditioning",
        "3bdrm",
        "4bdrm",
        "5bdrm",
        "6bdrm",
        "7bdrm",
        "8bdrm",
    ]

    regex_pattern = "|".join(search_terms)

    condition = (
        df["description"].str.contains(regex_pattern, case=False, na=False)
        | df["title"].str.contains(regex_pattern, case=False, na=False)
    ) & df["Self Contained"].isna()

    df.loc[condition, "Self Contained"] = "Yes"
    print(
        f"Set 'Self Contained' to 'Yes' for {condition.sum()} listings based on description/title and prior NaN status."
    )

    # 1. Rent is 2000 cedis +
    df.loc[
        (df["Self Contained"].isna()) & (df["price"] >= 2000),
        "Self Contained",
    ] = "Yes"

    # 2. Old condition AND Shared Apartment
    df.loc[
        (df["Self Contained"].isna())
        & (df["Condition"] == "Old")
        & (df["house_type"] == "Shared Apartment"),
        "Self Contained",
    ] = "No"

    # 3. Has amenities/furnishing
    df.loc[
        (df["Self Contained"].isna()) & (df["Wardrobe"] == 1)
        | (df["Wi-Fi"] == 1)
        | (df["TV"] == 1)
        | (df["Furnishing"] == "Furnished")
        | (df["Facilities"].notna())
        | (df["Dishwasher"] == 1)
        | (df["Dining Area"] == 1)
        | (df["Caution Fee"].notna()),
        "Self Contained",
    ] = "Yes"

    return df


df = clean_self_contained(df)

NAs in self contained feature: 8916
Set 'Self Contained' to 'Yes' for 8015 listings based on description/title and prior NaN status.


In [369]:
df[df["Self Contained"].isna()]

,url,description,fetch_date,title,location,house_type,bathrooms,bedrooms,price,Property Address,...,Kitchen Cabinets,Kitchen Shelf,Microwave,Pop Ceiling,Pre-Paid Meter,Refrigerator,TV,Tiled Floor,Wardrobe,Wi-Fi
113,https://jiji.com.gh/adenta-municipal/houses-ap...,Two bedroom apartment for rent*\nLocation: Ade...,2025-12-26 21:14:50.554243,"2bdrm Apartment in Danfa, Adenta for rent",Adenta,Apartment,3,2,1700.0,adenta Danfa,...,0,0,0,0,0,0,0,0,0,0
457,https://jiji.com.gh/north-industrial-area/hous...,"Tilled room,burglar proofs,fitted kitchen, spa...",2025-12-26 21:14:44.697253,"1bdrm Apartment in Walako Estate , North Indus...",North Industrial Area,Apartment,1,1,700.0,North Kaneshie,...,0,0,0,0,0,0,0,0,0,0
467,https://jiji.com.gh/tema-metropolitan/houses-a...,Features\n1 bedroom\n1 washroom\n1 living\nkit...,2025-12-26 21:14:44.690424,"1bdrm Apartment in Community 25, Tema Metropol...",Tema Metropolitan,Apartment,1,1,1500.0,community 25,...,0,0,0,0,0,0,0,0,0,0
515,https://jiji.com.gh/pillar-2/houses-apartments...,A 2 bedroom house for rent at Dome Pillar 2 (A...,2025-12-26 21:14:44.640495,"2bdrm Apartment in Paradise Estates, Pillar 2 ...",Dome,Apartment,2,2,1800.0,Dome Pillar 2,...,0,0,0,0,0,0,0,0,0,0
527,https://jiji.com.gh/gbawe/houses-apartments-fo...,2bedroom apartment around gbawe cp last top fo...,2025-12-26 21:14:44.637462,"2bdrm Apartment in Cp Last Top, Gbawe for rent",Gbawe,Apartment,2,2,1800.0,cp,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14620,https://jiji.com.gh/weija/houses-apartments-fo...,A very nice 2 Bedrooms apartment for rent at W...,2025-12-26 21:12:33.406965,2bdrm Apartment in Weija for rent,Weija,Apartment,1,2,1200.0,Oshieye,...,0,0,0,0,0,0,0,0,0,0
14640,https://jiji.com.gh/police-station-oyibi/house...,A decent 2bdrms apartment for rent for Ghc1500...,2025-12-26 21:12:33.105252,"2bdrm Apartment in Biosk Properties, Police St...",Oyibi,Apartment,2,2,1500.0,"Police station area, Oyibi",...,0,0,0,0,0,0,0,0,0,0
14655,https://jiji.com.gh/bortiano/houses-apartments...,"2 Bedrooms apartment for rent at Bortiano, a l...",2025-12-26 21:12:33.092778,2bdrm Apartment in Bortiano for rent,Weija,Apartment,2,2,1000.0,Adansiman,...,0,0,0,0,0,0,0,0,0,0
14680,https://jiji.com.gh/amasaman/houses-apartments...,One Bedroom studio apartment for rent at Amasa...,2025-12-26 21:12:32.468030,1bdrm Apartment in Amasaman for rent,Ga West Municipal,Apartment,1,1,1500.0,Amasaman,...,0,0,0,0,0,0,0,0,0,0


In [370]:
# df.to_csv(OUTPUT)